In [1]:
!pip uninstall -y torchao
!pip install -U transformers datasets accelerate peft trl bitsandbytes

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 158.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.0
    Uninstalling transformers-5.12.0:
      Successfully uninstalled transfo

In [43]:
!pip install -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 42.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.19.0
    Uninstalling huggingface_hub-1.19.0:
      Successfully uninstalled huggingface_hub-1.19.0


In [2]:
import json
import random
from pathlib import Path
from datasets import load_dataset
import random
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig


# Data formating

In [3]:
DATA_DIR = Path("data/no_robots_sft")
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
raw_data = load_dataset("HuggingFaceH4/no_robots")

README.md:   0%|          | 0.00/5.61k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/10.5M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/571k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

In [5]:
print(raw_data)

DatasetDict({
    train: Dataset({
        features: ['prompt', 'prompt_id', 'messages', 'category'],
        num_rows: 9500
    })
    test: Dataset({
        features: ['prompt', 'prompt_id', 'messages', 'category'],
        num_rows: 500
    })
})


In [6]:
train_raw = raw_data["train"].shuffle(seed=42)

In [7]:
test_raw = raw_data["test"].shuffle(seed=42)

In [8]:
train_raw[4]

{'prompt': 'As if you were Sir David Attenborough, write a narration following someone playing Counter Strike Global Offensive on the map Dust 2 executing a B-Site Rush. The player gets the bomb down but dies when the enemy retakes the site.',
 'prompt_id': 'f6c680386eb6d3b503e4c709dba9eb0ff3e4365b974ac77079ae9296bf8fdad7',
 'messages': [{'content': 'As if you were Sir David Attenborough, write a narration following someone playing Counter Strike Global Offensive on the map Dust 2 executing a B-Site Rush. The player gets the bomb down but dies when the enemy retakes the site.',
   'role': 'user'},
  {'content': 'Here we see the T-side player in his natural habitat. At his point in his development, he has many trials ahead, though the objective is clear. After an unfortunate stroke of hard luck, there is little in the way of the economy that he may use to prepare. The call is made for a desperate -- but not uncommon -- play. He purchases what little he can from the store and readies him

In [9]:
split = train_raw.train_test_split(test_size=500, seed=42)
train_split = split["train"]
valid_split = split["test"]

In [10]:
len(valid_split)

500

In [11]:
system_message = "You are a helpful assistant."

In [12]:
def convert_example(example):
    messages = example.get("messages", [])

    if not messages:
        return None

    cleaned = []

    for msg in messages:
        role = str(msg.get("role", "")).strip().lower()
        content = str(msg.get("content", "")).strip()

        if role not in {"system", "user", "assistant"}:
            return None

        if not content:
            return None

        cleaned.append({"role": role, "content": content})

    if cleaned[-1]["role"] != "assistant":
        return None

    if cleaned[0]["role"] != "system":# add system message if there is no in asample
        cleaned = [{"role": "system", "content": system_message}] + cleaned

    return {
        "messages": cleaned,
        "category": example.get("category", "unknown")
    }


In [13]:
def convert_split(ds):
    rows = []

    for ex in ds:
        item = convert_example(ex)
        if item is not None:
            rows.append(item)

    return rows

In [14]:
def save_jsonl(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

In [15]:
train_data = convert_split(train_split)
valid_data = convert_split(valid_split)
test_data = convert_split(test_raw)

print("converted train:", len(train_data))
print("converted valid:", len(valid_data))
print("converted test:", len(test_data))

save_jsonl(train_data, DATA_DIR / "train.jsonl")
save_jsonl(valid_data, DATA_DIR / "valid.jsonl")
save_jsonl(test_data, DATA_DIR / "test.jsonl")

converted train: 8999
converted valid: 500
converted test: 500


In [16]:
train_data[3]

{'messages': [{'role': 'system',
   'content': 'Bubba is a chatbot that gives the user ideas for different dishes to make but only provides ideas for dishes that are made with shrimp.'},
  {'role': 'user', 'content': 'I need something for dinner'},
  {'role': 'assistant',
   'content': 'Shrimp is great for dinner! You could have pineapple shrimp.'},
  {'role': 'user', 'content': "I don't want something sweet"},
  {'role': 'assistant',
   'content': "Shrimp doesn't always have to be sweet. You could try shrimp salad."},
  {'role': 'user', 'content': 'how about something spicy?'},
  {'role': 'assistant',
   'content': 'Shrimp can be spicy! You could try pepper shrimp.'}],
 'category': 'Chat'}

In [17]:
print(json.dumps(train_data[3], ensure_ascii=False, indent=2))

{
  "messages": [
    {
      "role": "system",
      "content": "Bubba is a chatbot that gives the user ideas for different dishes to make but only provides ideas for dishes that are made with shrimp."
    },
    {
      "role": "user",
      "content": "I need something for dinner"
    },
    {
      "role": "assistant",
      "content": "Shrimp is great for dinner! You could have pineapple shrimp."
    },
    {
      "role": "user",
      "content": "I don't want something sweet"
    },
    {
      "role": "assistant",
      "content": "Shrimp doesn't always have to be sweet. You could try shrimp salad."
    },
    {
      "role": "user",
      "content": "how about something spicy?"
    },
    {
      "role": "assistant",
      "content": "Shrimp can be spicy! You could try pepper shrimp."
    }
  ],
  "category": "Chat"
}


In [18]:
def validate_rows(rows, name):
    errors = []
    seen = set()

    for i, row in enumerate(rows):
        if "messages" not in row:
            errors.append((i, "missing messages"))
            continue

        messages = row["messages"]

        if not isinstance(messages, list) or len(messages) < 3:
            errors.append((i, "bad messages length"))
            continue

        if messages[0]["role"] != "system":
            errors.append((i, "first message is not system"))

        if messages[-1]["role"] != "assistant":
            errors.append((i, "last message is not assistant"))

        roles = [m["role"] for m in messages]

        if "user" not in roles:
            errors.append((i, "missing user"))

        if "assistant" not in roles:
            errors.append((i, "missing assistant"))

        for m in messages:
            if m["role"] not in {"system", "user", "assistant"}:
                errors.append((i, f"bad role: {m['role']}"))

            if not str(m["content"]).strip():
                errors.append((i, "empty content"))

        key = json.dumps(messages, ensure_ascii=False)
        if key in seen:
            errors.append((i, "duplicate"))
        seen.add(key)

    print(f"{name} errors:", len(errors))

    for err in errors[:10]:
        print(err)


validate_rows(train_data, "train")
validate_rows(valid_data, "valid")
validate_rows(test_data, "test")

train errors: 0
valid errors: 0
test errors: 0


In [19]:
random.seed(42)

def make_eval_item(example):
    messages = example["messages"]

    return {
        "prompt_messages": messages[:-1],
        "target_answer": messages[-1]["content"],
        "category": example.get("category", "unknown")
    }

eval_50 = [
    make_eval_item(ex)
    for ex in random.sample(test_data, min(50, len(test_data)))
]

save_jsonl(eval_50, DATA_DIR / "eval_50.jsonl")

print("Saved eval_50:", len(eval_50))
print(json.dumps(eval_50[0], ensure_ascii=False, indent=2))

Saved eval_50: 50
{
  "prompt_messages": [
    {
      "role": "system",
      "content": "You are a helpful assistant."
    },
    {
      "role": "user",
      "content": "I find that I am a terrible procrastinator. I would like to manage this. Can you give me some advice on how to overcome this issue? I want exactly four ways or methods. Keep it brief."
    }
  ],
  "target_answer": "1. Make decisions ahead of time and form routines to help you reduce the number of decisions you need to make during the day.\n2. Plan your days in advance to finish them before they begin.\n3. Attempt \"The Nothing Alternative\" by blocking out time for either working on your project or doing nothing at all.\n4. Determine the next physical step you must do to advance toward completion, then perform it. This will help you keep your attention on the present.",
  "category": "Generation"
}


In [20]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [21]:
def count_tokens(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

    return len(tokenizer(text)["input_ids"])


lengths = [count_tokens(ex) for ex in train_data]

print("Num examples:", len(lengths))
print("Min:", min(lengths))
print("Max:", max(lengths))
print("Avg:", sum(lengths) / len(lengths))
print("Over 512:", sum(x > 512 for x in lengths))
print("Over 1024:", sum(x > 1024 for x in lengths))
print("Over 1536:", sum(x > 1536 for x in lengths))
print("Over 2048:", sum(x > 2048 for x in lengths))

Num examples: 8999
Min: 40
Max: 6740
Avg: 298.5093899322147
Over 512: 1054
Over 1024: 153
Over 1536: 49
Over 2048: 20


Evaluate Qwen2.5 7B before training

In [22]:
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_OUT_PATH = OUT_DIR / "base_qwen2_5_7b_no_robots_eval_outputs.jsonl"

In [23]:
model_id = "Qwen/Qwen2.5-7B-Instruct"

In [24]:
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token

In [25]:
tokenizer.padding_side = "left"

In [26]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
    trust_remote_code=True
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [27]:
print("Device:", next(model.parameters()).device)
print("Dtype:", next(model.parameters()).dtype)

Device: cuda:0
Dtype: torch.bfloat16


In [28]:
def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

eval_rows = read_jsonl(DATA_DIR / "eval_50.jsonl")

print("Eval samples:", len(eval_rows))
print(eval_rows[0].keys())
print(eval_rows[0]["prompt_messages"])

Eval samples: 50
dict_keys(['prompt_messages', 'target_answer', 'category'])
[{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': 'I find that I am a terrible procrastinator. I would like to manage this. Can you give me some advice on how to overcome this issue? I want exactly four ways or methods. Keep it brief.'}]


In [29]:
def batched_generate(prompt_messages_list, batch_size=4, max_new_tokens=256):
    all_answers = []

    for start in range(0, len(prompt_messages_list), batch_size):
        batch_messages = prompt_messages_list[start:start + batch_size]

        texts = [
            tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
            for messages in batch_messages
        ]

        inputs = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048,
        ).to(model.device)

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
            )

        input_len = inputs["input_ids"].shape[1]

        for i in range(outputs.shape[0]):
            generated_tokens = outputs[i][input_len:]
            answer = tokenizer.decode(
                generated_tokens,
                skip_special_tokens=True
            ).strip()
            all_answers.append(answer)

        print(f"Done {min(start + batch_size, len(prompt_messages_list))}/{len(prompt_messages_list)}")

    return all_answers

In [30]:
prompt_messages_list = [item["prompt_messages"]for item in eval_rows]

answers = batched_generate(
    prompt_messages_list,
    batch_size=4,
    max_new_tokens=256
)

Done 4/50
Done 8/50
Done 12/50
Done 16/50
Done 20/50
Done 24/50
Done 28/50
Done 32/50
Done 36/50
Done 40/50
Done 44/50
Done 48/50
Done 50/50


In [31]:
with open(BASE_OUT_PATH, "w", encoding="utf-8") as f:
    for idx, (item, pred) in enumerate(zip(eval_rows, answers)):
        row = {
            "id": idx,
            "prompt_messages": item["prompt_messages"],
            "target_answer": item["target_answer"],
            "model_answer": pred,
            "category": item.get("category", "unknown")
        }

        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Saved baseline outputs to:", BASE_OUT_PATH)

Saved baseline outputs to: outputs/base_qwen2_5_7b_no_robots_eval_outputs.jsonl


In [33]:
def preview_outputs(path, n=10):
    rows = read_jsonl(path)

    for i in range(min(n, len(rows))):
        item = rows[i]

        print("=" * 100)
        print("ID:", item["id"])
        print("CATEGORY:", item.get("category", "unknown"))

        print("\nPROMPT:")
        for msg in item["prompt_messages"]:
            print(f"{msg['role'].upper()}:")
            print(msg["content"][:1200])
            print()

        print("TARGET:")
        print(item["target_answer"][:1200])

        print("\nBASE MODEL ANSWER:")
        print(item["model_answer"][:1200])

preview_outputs(BASE_OUT_PATH, n=10)

ID: 0
CATEGORY: Generation

PROMPT:
SYSTEM:
You are a helpful assistant.

USER:
I find that I am a terrible procrastinator. I would like to manage this. Can you give me some advice on how to overcome this issue? I want exactly four ways or methods. Keep it brief.

TARGET:
1. Make decisions ahead of time and form routines to help you reduce the number of decisions you need to make during the day.
2. Plan your days in advance to finish them before they begin.
3. Attempt "The Nothing Alternative" by blocking out time for either working on your project or doing nothing at all.
4. Determine the next physical step you must do to advance toward completion, then perform it. This will help you keep your attention on the present.

BASE MODEL ANSWER:
Sure! Here are four effective methods to help you overcome procrastination:

1. **Set Clear Goals**: Break your tasks into smaller, manageable goals and set specific deadlines for each.

2. **Use the Pomodoro Technique**: Work in focused 25-minute in

# Initialization Model With LORA & Training

In [34]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

DATA_DIR = Path("data/no_robots_sft")
TRAIN_PATH = DATA_DIR / "train.jsonl"
VALID_PATH = DATA_DIR / "valid.jsonl"

OUTPUT_DIR = "outputs/qwen2_5_7b_no_robots_lora_r32_bf16"

print("Train path:", TRAIN_PATH)
print("Valid path:", VALID_PATH)
print("GPU:", torch.cuda.get_device_name(0))

Train path: data/no_robots_sft/train.jsonl
Valid path: data/no_robots_sft/valid.jsonl
GPU: NVIDIA A100-SXM4-40GB


In [35]:
model.config.use_cache = False

In [36]:
train_dataset = load_dataset(
    "json",
    data_files=str(TRAIN_PATH),
    split="train"
)

Generating train split: 0 examples [00:00, ? examples/s]

In [37]:
print(train_dataset)

Dataset({
    features: ['messages', 'category'],
    num_rows: 8999
})


In [38]:
eval_dataset = load_dataset(
    "json",
    data_files=str(VALID_PATH),
    split="train"
)

Generating train split: 0 examples [00:00, ? examples/s]

In [39]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)

In [40]:
args = SFTConfig(
    output_dir=OUTPUT_DIR,

    max_length=2048,
    packing=False,

    num_train_epochs=1,
    learning_rate=1e-4,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,

    logging_steps=20,

    eval_strategy="epoch",
    save_strategy="epoch",

    bf16=True,
    fp16=False,

    optim="adamw_torch",
    report_to="none",

    assistant_only_loss=True,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

/tmp/ipykernel_1962/2399789736.py:1: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  args = SFTConfig(


In [41]:
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
)

Tokenizing train dataset:   0%|          | 0/8999 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

In [42]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,1.814437,1.773123,1.771909,0.580124,2674509.000000


TrainOutput(global_step=563, training_loss=1.8046449141222884, metrics={'train_runtime': 1907.4308, 'train_samples_per_second': 4.718, 'train_steps_per_second': 0.295, 'total_flos': 2.0432963017874534e+17, 'train_loss': 1.8046449141222884, 'epoch': 1.0})

In [2]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

NameError: name 'trainer' is not defined

#Test The Model after Trining

In [46]:
DATA_DIR = Path("data/no_robots_sft")
EVAL_PATH = DATA_DIR / "eval_50.jsonl"

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FT_OUT_PATH = OUT_DIR / "ft_qwen2_5_7b_no_robots_eval_outputs.jsonl"

model = trainer.model
model.eval()

tokenizer.padding_side = "left"

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


In [47]:
def batched_generate(prompt_messages_list, batch_size=4, max_new_tokens=512):
    all_answers = []

    for start in range(0, len(prompt_messages_list), batch_size):
        batch_messages = prompt_messages_list[start:start + batch_size]

        texts = [
            tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
            for messages in batch_messages
        ]

        inputs = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048,
        ).to(model.device)

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
            )

        input_len = inputs["input_ids"].shape[1]

        for i in range(outputs.shape[0]):
            generated_tokens = outputs[i][input_len:]
            answer = tokenizer.decode(
                generated_tokens,
                skip_special_tokens=True
            ).strip()
            all_answers.append(answer)

        print(f"Done {min(start + batch_size, len(prompt_messages_list))}/{len(prompt_messages_list)}")

    return all_answers


In [48]:
eval_rows = read_jsonl(EVAL_PATH)

prompt_messages_list = [
    item["prompt_messages"]
    for item in eval_rows
]

answers = batched_generate(
    prompt_messages_list,
    batch_size=4,
    max_new_tokens=512
)

Done 4/50
Done 8/50
Done 12/50
Done 16/50
Done 20/50
Done 24/50
Done 28/50
Done 32/50
Done 36/50
Done 40/50
Done 44/50
Done 48/50
Done 50/50


In [49]:
with open(FT_OUT_PATH, "w", encoding="utf-8") as f:
    for idx, (item, pred) in enumerate(zip(eval_rows, answers)):
        row = {
            "id": idx,
            "prompt_messages": item["prompt_messages"],
            "target_answer": item["target_answer"],
            "model_answer": pred,
            "category": item.get("category", "unknown")
        }

        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Saved fine-tuned outputs to:", FT_OUT_PATH)

Saved fine-tuned outputs to: outputs/ft_qwen2_5_7b_no_robots_eval_outputs.jsonl


In [50]:
BASE_PATH = Path("/content/outputs/base_qwen2_5_7b_no_robots_eval_outputs.jsonl")

In [54]:
base_rows = read_jsonl(BASE_PATH)
ft_rows = read_jsonl(FT_OUT_PATH)

def print_comparison(start=0, n=5):
    for i in range(start, min(start + n, len(ft_rows))):
        base = base_rows[i]
        ft = ft_rows[i]

        print("=" * 120)
        print("ID:", i)
        print("CATEGORY:", ft.get("category", "unknown"))

        print("\nPROMPT:")
        for msg in ft["prompt_messages"]:
            print(f"{msg['role'].upper()}:")
            print(msg["content"])
            print()

        print("TARGET:")
        print(ft["target_answer"])

        print("\nBASE MODEL ANSWER:")
        print(base["model_answer"])

        print("\nFINE-TUNED MODEL ANSWER:")
        print(ft["model_answer"])


print_comparison(start=0, n=10)

ID: 0
CATEGORY: Generation

PROMPT:
SYSTEM:
You are a helpful assistant.

USER:
I find that I am a terrible procrastinator. I would like to manage this. Can you give me some advice on how to overcome this issue? I want exactly four ways or methods. Keep it brief.

TARGET:
1. Make decisions ahead of time and form routines to help you reduce the number of decisions you need to make during the day.
2. Plan your days in advance to finish them before they begin.
3. Attempt "The Nothing Alternative" by blocking out time for either working on your project or doing nothing at all.
4. Determine the next physical step you must do to advance toward completion, then perform it. This will help you keep your attention on the present.

BASE MODEL ANSWER:
Sure! Here are four effective methods to help you overcome procrastination:

1. **Set Clear Goals**: Break your tasks into smaller, manageable goals and set specific deadlines for each.

2. **Use the Pomodoro Technique**: Work in focused 25-minute in

In [55]:
random.seed(7)

sample_ids = random.sample(range(len(ft_rows)), 10)

for i in sample_ids:
    print_comparison(start=i, n=1)

ID: 20
CATEGORY: Generation

PROMPT:
SYSTEM:
You are a helpful assistant.

USER:
Write a short story about a boy who finds a dog and keeps him.

TARGET:
It was a sunny summer day in the South and Jimmy was walking home from the park. As Jimmy turned the corner to his house he felt a nudge at his calf. He jumped, startled by the nudge, and looked down to see a cute, hungry, and dirty puppy. He couldn't believe it. He had always wanted a puppy of his own and this puppy appeared to be a stray. Jimmy scooped up the puppy and continued on his way home. 

When he arrived home, he hid the puppy under his jacket and snuck upstairs to the bathroom to get him cleaned up. He didn't want his mom to see the puppy, especially dirty and smelly. The puppy splashed around in the bath and shook water all over the whole room, soaking Jimmy in the process. 

"I'll name you Splashers!" exclaimed Jimmy. 

Soon after, Jimmy's mom heard the commotion and came to investigate, finding a damp Jimmy and a sopping